# Synthetic PDF To Database Demo

This is the only notebook demo in the release bundle. It generates a synthetic article PDF, extracts the embedded article table, writes a local SQLite database, and runs the complete reconstruction workflow. No real article text, private data, API key, or copyrighted table is used.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RUN_ROOT = ROOT
# If this project folder is read-only in your runtime, set RUN_ROOT to a writable path.
# RUN_ROOT = Path("C:/temp/arsenic_synthetic_demo")
sys.path.insert(0, str(ROOT / "src"))

from arsenic_workflow.devtools.generate_synthetic_pdf import generate_synthetic_pdf
from arsenic_workflow.synthetic_pdf import (
    PDF_RELATIVE_PATH,
    SQLITE_RELATIVE_PATH,
    OUTPUT_RELATIVE_DIR,
    extract_text_from_pdf,
    extract_synthetic_table,
    prepare_synthetic_database,
    prepare_synthetic_database_with_qwen,
    run_synthetic_pdf_demo,
)
from arsenic_workflow.io import read_table

dashscope_configured = bool(os.getenv("DASHSCOPE_API_KEY"))
dashscope_configured


The API key is checked only as an environment variable. To inspect it in PowerShell, use `echo $env:DASHSCOPE_API_KEY`. This notebook does not print the key value and does not require network access.

## 2. Generate Synthetic PDF

In [ ]:
pdf_path = generate_synthetic_pdf(RUN_ROOT / PDF_RELATIVE_PATH)
pdf_path


## 3. Extract The Synthetic Article Table

In [ ]:
pdf_text = extract_text_from_pdf(pdf_path)
candidate_table = extract_synthetic_table(pdf_text)
candidate_table.head()


## 4. Write SQLite Database

In [ ]:
USE_QWEN = False
PROMPT_TEMPLATE = Path("templates") / "prompt_v1.txt"
QWEN_RUNS = 2
prepared = prepare_synthetic_database_with_qwen(RUN_ROOT, prompt_template_path=PROMPT_TEMPLATE, runs=QWEN_RUNS) if USE_QWEN else prepare_synthetic_database(RUN_ROOT)
prepared


The database is written to a relative path inside the release bundle.

In [ ]:
RUN_ROOT / SQLITE_RELATIVE_PATH


## 5. Run Complete Reconstruction Workflow

In [ ]:
outputs = run_synthetic_pdf_demo(RUN_ROOT, use_qwen=USE_QWEN, prepare=False, prompt_template_path=PROMPT_TEMPLATE, qwen_runs=QWEN_RUNS)
outputs


## 6. Inspect Outputs

In [ ]:
model_ready = read_table(outputs["model_ready_records"])
model_ready[["record_id", "scientific_name", "arsenic_mg_per_kg_ww", "latitude", "longitude", "env_match_distance_km"]]


In [ ]:
read_table(outputs["validation_metrics"])


In [ ]:
read_table(outputs["speciation_summary"]).head(12)
